# Tema de Programare: Construirea Modelelor Bazate pe Arbori de la Zero

Bun venit la tema despre Modele Bazate pe Arbori.

Metodele bazate pe arbori se numara printre cei mai populari si mai puternici algoritmi de machine learning atat pentru clasificare, cat si pentru regresie. De la arbori de decizie simpli pana la metode ensemble sofisticate precum Random Forest si Gradient Boosting, acesti algoritmi si-au demonstrat valoarea in nenumarate aplicatii din lumea reala, inclusiv prin castigarea multor competitii Kaggle.

In aceasta tema, vei implementa clasificatori bazati pe arbori folosind atat scikit-learn, cat si implementari de la zero cu NumPy. Vei explora arbori individuali, le vei intelege limitarile, iar apoi vei invata cum metodele ensemble depasesc aceste slabiciuni pentru a crea modele state-of-the-art.

Arborii de decizie functioneaza impartind recursiv datele pe baza valorilor trasaturilor, creand o structura ierarhica foarte interpretabila. Fiecare split urmareste sa maximizeze puritatea submultimilor rezultate. Desi arborii individuali pot face overfitting, combinarea mai multor arbori prin bagging (Random Forest) sau boosting (XGBoost, LightGBM) creeaza modele atat puternice, cat si robuste.

**Ce vei face in aceasta tema**

* Vei intelege fundamentul matematic al arborilor de decizie, inclusiv entropia, impuritatea Gini si information gain.
* Vei construi si vizualiza arbori de decizie, interpretand regulile invatate.
* Vei controla overfitting-ul prin tehnici de pruning (`max_depth`, `min_samples_split`).
* Vei implementa Random Forest si vei intelege cum bootstrap aggregating reduce varianta.
* Vei analiza eroarea out-of-bag si vei extrage importanta trasaturilor din ensemble-uri.
* Vei antrena modele Gradient Boosting (XGBoost, LightGBM) cu early stopping.
* Vei compara bagging vs boosting si vei intelege punctele forte diferite ale fiecaruia.
* Vei construi un arbore de decizie complet de la zero, cu split-uri bazate pe Gini/entropie.
* Vei implementa Random Forest de la zero, cu bootstrap sampling si votare.

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI:</h4>

* Toate celulele sunt inghetate, cu exceptia celor in care trebuie sa trimiti solutiile tale sau atunci cand este mentionat explicit ca poti interactiona cu ele.

* In fiecare celula de exercitiu, cauta comentariile `### ÎNCEPUT DE COD AICI ###` si `### SFÂRȘIT DE COD AICI ###`. Acestea iti arata unde trebuie sa scrii codul-solutie. **Nu adauga si nu modifica niciun cod din afara acestor comentarii**.

* Poti adauga celule noi pentru experimente, dar acestea vor fi omise de grader, asa ca nu te baza pe celulele nou create pentru a gazdui codul-solutie; foloseste locurile deja oferite pentru asta.

* Evita sa folosesti variabile globale daca nu este absolut necesar. Grader-ul iti testeaza codul intr-un mediu izolat, fara sa ruleze toate celulele de sus in jos. Ca rezultat, variabilele globale pot fi indisponibile in momentul evaluarii. Variabilele globale care trebuie folosite vor fi definite cu MAJUSCULE.

* Pentru a trimite notebook-ul la evaluare, salveaza-l mai intai facand click pe iconita 💾 din coltul stanga-sus al paginii, apoi apasa pe butonul `Submit assignment` din coltul dreapta-sus al paginii.
---


## Cuprins
- [Importuri](#imports)
- [1 - Introducere in arborii de decizie](#1)
    - [1.1 - Cum functioneaza arborii de decizie](#1-1)
    - [1.2 - Criterii de split: Gini vs Entropie](#1-2)
    - [1.3 - Overfitting in arborii de decizie](#1-3)
- [2 - Antrenarea si vizualizarea arborilor de decizie](#2)
    - [2.1 - Construirea primului tau arbore de decizie](#2-1)
    - **[Exercitiul 1 - train_decision_tree](#ex-1)**
    - [2.2 - Vizualizarea si interpretarea arborilor](#2-2)
- [3 - Pruning si regularizare](#3)
    - **[Exercitiul 2 - compare_tree_depths](#ex-2)**
- [4 - Ensemble-ul Random Forest](#4)
    - [4.1 - Intelegerea bootstrap aggregating](#4-1)
    - **[Exercitiul 3 - train_random_forest](#ex-3)**
    - [4.2 - Analiza importantei trasaturilor](#4-2)
- [5 - Gradient Boosting](#5)
    - [5.1 - Cum difera boosting-ul de bagging](#5-1)
    - **[Exercitiul 4 - train_gradient_boosting](#ex-4)**
- [6 - Comparatia metodelor ensemble](#6)
    - **[Exercitiul 5 - compare_ensemble_methods](#ex-5)**
- [7 - Arbore de decizie de la zero](#7)
    - [7.1 - Implementarea criteriilor de split](#7-1)
    - **[Exercitiul 6a - calculate_gini](#ex-6a)**
    - **[Exercitiul 6b - calculate_entropy](#ex-6b)**
    - **[Exercitiul 6c - find_best_split](#ex-6c)**
    - **[Exercitiul 6d - build_decision_tree_scratch](#ex-6d)**
- [8 - Random Forest de la zero](#8)
    - **[Exercitiul 7 - random_forest_from_scratch](#ex-7)**


<a name='imports'></a>
## Importuri


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests

from sklearn.datasets import load_iris, load_wine, make_classification, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# For advanced boosting (install if needed: pip install xgboost lightgbm)
try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available. Install with: pip install lightgbm")

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

<a name='1'></a>
## 1 - Introducere in arborii de decizie

Arborii de decizie sunt modele intuitive si interpretabile care invata o ierarhie de reguli if-then-else din date. Ei pot gestiona atat trasaturi numerice, cat si categorice, necesita preprocesare minima a datelor si surprind in mod natural relatii neliniare.


<a name='1-1'></a>
### 1.1 - Cum functioneaza arborii de decizie

Un arbore de decizie partitioneaza recursiv spatiul trasaturilor in regiuni. La fiecare nod, algoritmul:

1. **Evalueaza toate split-urile posibile**: Pentru fiecare trasatura si fiecare valoare de prag
2. **Alege cel mai bun split**: Pe baza unui criteriu de puritate (Gini sau entropie)
3. **Creeaza noduri copil**: Repeta recursiv pana cand sunt indeplinite criteriile de oprire

Predictia pentru un exemplu se obtine urmand calea de decizie de la radacina la frunza, apoi folosind clasa majoritara (clasificare) sau valoarea medie (regresie) a exemplelor de antrenare din acea frunza.

**Proprietati cheie:**
- **Neparametric**: Nu face presupuneri despre distributia datelor
- **Neliniare**: Poate surprinde frontiere de decizie complexe
- **Interpretabil**: Usor de vizualizat si explicat
- **Predispus la overfitting**: Poate memora datele de antrenare daca nu este regularizat


<a name='1-2'></a>
### 1.2 - Criterii de split: Gini vs Entropie

Calitatea unui split este masurata prin cat de mult creste "puritatea" nodurilor copil rezultate. Doua criterii uzuale sunt:

**Impuritatea Gini**: Masoara probabilitatea de a clasifica gresit un element ales aleator:

$$\text{Gini}(S) = 1 - \sum_{i=1}^{C} p_i^2$$

unde $p_i$ este proportia clasei $i$ in setul $S$, iar $C$ este numarul de clase.

**Entropia (Information Gain)**: Masoara continutul mediu de informatie:

$$\text{Entropy}(S) = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

**Information Gain** obtinut dintr-un split:

$$\text{IG}(S, A) = \text{Entropy}(S) - \sum_{v \in A} \frac{|S_v|}{|S|} \text{Entropy}(S_v)$$

unde $A$ este atributul (trasatura), iar $S_v$ sunt submultimile create prin impartirea dupa $A$.

Ambele criterii functioneaza similar in practica:
- **Gini**: Mai rapid de calculat (fara logaritm), tinde sa izoleze clasa cea mai frecventa
- **Entropie**: Split-uri mai echilibrate, fundamentate teoretic in teoria informatiei

**Nod pur**: Gini = 0, Entropie = 0 (toate exemplele apartin unei singure clase)
**Impuritate maxima**: Gini = 0.5 (binar), Entropie = 1.0 (binar) cand clasele sunt distribuite uniform


<a name='1-3'></a>
### 1.3 - Overfitting in arborii de decizie

Fara constrangeri, un arbore de decizie va creste pana cand fiecare frunza contine exemple dintr-o singura clasa (sau un singur exemplu). Acest lucru creeaza un model care se potriveste perfect pe datele de train, dar generalizeaza slab.

**Semne ale overfitting-ului:**
- Acuratetea pe train ≈ 100%, dar acuratetea pe test este mult mai mica
- Arbore foarte adanc, cu multe frunze
- Frontiere de decizie complexe care urmaresc zgomotul

**Tehnici de regularizare:**
- **max_depth**: Limiteaza adancimea arborelui
- **min_samples_split**: Numarul minim de exemple necesar pentru a split-ui un nod
- **min_samples_leaf**: Numarul minim de exemple necesar in fiecare frunza
- **max_leaf_nodes**: Limiteaza numarul total de frunze
- **min_impurity_decrease**: Imbunatatirea minima necesara pentru un split

**Strategii de pruning:**
- **Pre-pruning**: Opreste cresterea mai devreme (folosind constrangerile de mai sus)
- **Post-pruning**: Creste arborele complet, apoi elimina ramurile care nu imbunatatesc performanta pe validare


<a name='helpers'></a>
## Functii ajutatoare

Vom defini functii utilitare pentru vizualizare si analiza.


In [3]:
def plot_decision_boundary_2d(model, X, y, title="Decision Boundary", h=0.02):
    """Plot decision boundary for 2D data."""
    X = np.asarray(X)
    y = np.asarray(y)
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 7))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    scatter = plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', s=50, cmap='viridis')
    plt.colorbar(scatter)
    plt.title(title)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

def visualize_tree(model, feature_names=None, class_names=None, max_depth=3):
    """Visualize decision tree structure."""
    plt.figure(figsize=(20, 10))
    plot_tree(model, 
              feature_names=feature_names,
              class_names=class_names,
              filled=True, 
              rounded=True,
              fontsize=10,
              max_depth=max_depth)
    plt.title("Decision Tree Structure")
    plt.show()

<a name='2'></a>
## 2 - Antrenarea si vizualizarea arborilor de decizie

Hai sa construim primul nostru arbore de decizie si sa exploram modul in care partitioneaza spatiul trasaturilor.


<a name='2-1'></a>
### 2.1 - Construirea primului tau arbore de decizie

Vom folosi setul de date Iris, o problema clasica de clasificare multi-clasa cu 3 clase si 4 trasaturi.


In [4]:
# Load Iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"Dataset shape: {X_iris.shape}")
print(f"Number of classes: {len(np.unique(y_iris))}")
print(f"Class names: {iris.target_names}")
print(f"Feature names: {iris.feature_names}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

Dataset shape: (150, 4)
Number of classes: 3
Class names: ['setosa' 'versicolor' 'virginica']
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Training samples: 105
Testing samples: 45


<a name='ex-1'></a>
#### **Exercitiul 1 - `train_decision_tree`**

**Sarcina ta:**

Implementeaza functia `train_decision_tree` care antreneaza un clasificator de tip arbore de decizie cu hiperparametrii specificati.

Trebuie sa implementezi:

* **Creeaza `DecisionTreeClassifier`**:
    * Foloseste `DecisionTreeClassifier` din scikit-learn.
    * Seteaza `criterion` (criteriul de split: `'gini'` sau `'entropy'`).
    * Seteaza `max_depth` pentru a controla adancimea arborelui.
    * Seteaza `min_samples_split` pentru a controla cand are loc split-ul.
    * Seteaza `random_state=42` pentru reproductibilitate.

* **Antreneaza modelul**:
    * Antreneaza pe datele de train oferite.

* **Returneaza modelul antrenat**:
    * Returneaza clasificatorul deja antrenat.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru crearea arborelui:**
* Importul este deja facut: `from sklearn.tree import DecisionTreeClassifier`.
* Creeaza modelul: `model = DecisionTreeClassifier(criterion=criterion, max_depth=max_depth, min_samples_split=min_samples_split, random_state=42)`.

**Pentru antrenare:**
* Antreneaza: `model.fit(X, y)`.

**Pentru returnare:**
* Returneaza modelul deja antrenat.

</details>


In [ ]:
# GRADED FUNCTION: train_decision_tree
def train_decision_tree(X, y, criterion='gini', max_depth=None, min_samples_split=2):
    """
    Train a decision tree classifier.

    Args:
        X: Training features of shape (n_samples, n_features)
        y: Training labels of shape (n_samples,)
        criterion: Split criterion - 'gini' or 'entropy'
        max_depth: Maximum depth of tree (None = unlimited)
        min_samples_split: Minimum samples required to split a node

    Returns:
        model: Trained DecisionTreeClassifier
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Create DecisionTreeClassifier with specified parameters
    
    # Fit the model
    
    ### SFÂRȘIT DE COD AICI ###
    
    return model

In [ ]:
# Test your implementation
tree_model = train_decision_tree(X_train, y_train, criterion='gini', max_depth=3)

# Make predictions
y_pred = tree_model.predict(X_test)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

print(f"\nTree depth: {tree_model.get_depth()}")
print(f"Number of leaves: {tree_model.get_n_leaves()}")
print(f"Number of features used: {np.sum(tree_model.feature_importances_ > 0)}")

##### **Rezultatul asteptat**
```
Test Accuracy: 0.9778

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       1.00      0.93      0.97        15
   virginica       0.94      1.00      0.97        15

    accuracy                           0.98        45
   macro avg       0.98      0.98      0.98        45
weighted avg       0.98      0.98      0.98        45

Tree depth: 3
Number of leaves: 5
Number of features used: 3
```


In [ ]:
unittests.exercise_1(train_decision_tree)

<a name='2-2'></a>
### 2.2 - Vizualizarea si interpretarea arborilor

Unul dintre cele mai mari puncte forte ale arborilor de decizie este interpretabilitatea lor. Hai sa vizualizam structura arborelui si sa intelegem regulile invatate.


In [ ]:
# Visualize the tree structure
visualize_tree(tree_model, 
               feature_names=iris.feature_names,
               class_names=iris.target_names,
               max_depth=3)

print("\nHow to read the tree:")
print("• Each box is a node")
print("• 'gini' or 'entropy': impurity at that node")
print("• 'samples': number of training samples reaching that node")
print("• 'value': class distribution [class0, class1, class2]")
print("• 'class': predicted class (majority)")
print("• Color intensity: purity (darker = more pure)")

In [ ]:
# Extract feature importance
importances = tree_model.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Feature Importances in Decision Tree")
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [iris.feature_names[i] for i in indices], rotation=45, ha='right')
plt.ylabel("Importance")
plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
for i, idx in enumerate(indices):
    print(f"{i+1}. {iris.feature_names[idx]}: {importances[idx]:.4f}")

In [ ]:
# For 2D visualization, use only 2 features
X_train_2d = X_train[:, [2, 3]]  # petal length and width
X_test_2d = X_test[:, [2, 3]]

tree_2d = train_decision_tree(X_train_2d, y_train, criterion='gini', max_depth=3)
plot_decision_boundary_2d(tree_2d, X_train_2d, y_train, 
                         title="Decision Tree Boundary (Petal Length vs Width)")

print(f"Accuracy with 2 features: {accuracy_score(y_test, tree_2d.predict(X_test_2d)):.4f}")

<a name='3'></a>
## 3 - Pruning si regularizare

Hai sa exploram modul in care hiperparametri diferiti influenteaza complexitatea arborelui si performanta, demonstrand compromisul bias-variance.


<a name='ex-2'></a>
#### **Exercitiul 2 - `compare_tree_depths`**

**Sarcina ta:**

Implementeaza functia `compare_tree_depths` care antreneaza arbori de decizie cu valori diferite pentru `max_depth` si compara performanta lor.

Trebuie sa implementezi:

* **Parcurge valorile de adancime**:
    * Pentru fiecare valoare `max_depth` din lista oferita.
    * Antreneaza un arbore de decizie cu acea adancime.

* **Evalueaza pe train si test**:
    * Calculeaza acuratetea pe train pentru a detecta overfitting-ul.
    * Calculeaza acuratetea pe test pentru a masura generalizarea.

* **Stocheaza rezultatele**:
    * Inregistreaza adancimea, acuratetea pe train, acuratetea pe test, numarul de frunze si adancimea efectiva a arborelui.
    * Returneaza rezultatele sub forma de DataFrame pandas pentru analiza usoara.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru iterarea prin adancimi:**
* Itereaza: `for depth in max_depths:`.
* Antreneaza: `model = train_decision_tree(X_train, y_train, max_depth=depth)`.

**Pentru evaluare:**
* Acuratete train: `train_acc = accuracy_score(y_train, model.predict(X_train))`.
* Acuratete test: `test_acc = accuracy_score(y_test, model.predict(X_test))`.
* Statistici arbore: `model.get_depth()`, `model.get_n_leaves()`.

**Pentru stocarea rezultatelor:**
* Adauga intr-o lista: `results.append({'max_depth': depth, 'train_acc': train_acc, ...})`.
* Converteste in DataFrame: `df = pd.DataFrame(results)`.

</details>


In [ ]:
# GRADED FUNCTION: compare_tree_depths
def compare_tree_depths(X_train, y_train, X_test, y_test, max_depths=None):
    """
    Train decision trees with different max_depth values and compare performance.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        max_depths: List of max_depth values to try

    Returns:
        results_df: DataFrame with columns: max_depth, train_acc, test_acc, n_leaves, tree_depth
    """
    if max_depths is None:
        max_depths = [1, 2, 3, 4, 5, 10, 15, 20, None]
    
    results = []
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Loop through each max_depth value
    
        # Train model with this max_depth
        
        # Evaluate on train and test
        
        # Get tree statistics
        
        # Store results
        
    # Convert to DataFrame
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results_df

In [ ]:
# Test your implementation
depth_results = compare_tree_depths(X_train, y_train, X_test, y_test)

print("Tree Depth Comparison:")
print(depth_results.to_string(index=False))

# Visualize the results
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Accuracy vs Depth
axes[0].plot(depth_results['max_depth'].fillna(25), depth_results['train_acc'], 
             marker='o', label='Train Accuracy', linewidth=2)
axes[0].plot(depth_results['max_depth'].fillna(25), depth_results['test_acc'], 
             marker='s', label='Test Accuracy', linewidth=2)
axes[0].set_xlabel('Max Depth')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs Tree Depth')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Tree Complexity vs Depth
axes[1].plot(depth_results['max_depth'].fillna(25), depth_results['n_leaves'], 
             marker='o', label='Number of Leaves', linewidth=2)
axes[1].plot(depth_results['max_depth'].fillna(25), depth_results['tree_depth'], 
             marker='s', label='Actual Tree Depth', linewidth=2)
axes[1].set_xlabel('Max Depth')
axes[1].set_ylabel('Count')
axes[1].set_title('Tree Complexity vs Max Depth')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey Observations:")
print("• Training accuracy increases with depth (can overfit)")
print("• Test accuracy plateaus or decreases after optimal depth")
print("• Deeper trees have more leaves (higher complexity)")

##### **Rezultatul asteptat**
```
Tree Depth Comparison:
 max_depth  train_acc  test_acc  n_leaves  tree_depth
       1.0   0.666667  0.644444         2           1
       2.0   0.971429  0.933333         5           2
       3.0   0.980952  0.977778         5           3
       ...
```


In [ ]:
unittests.exercise_2(compare_tree_depths)

<a name='4'></a>
## 4 - Ensemble-ul Random Forest

Un singur arbore de decizie este instabil: schimbari mici in datele de train pot duce la arbori complet diferiti. Random Forest rezolva aceasta problema combinand multi arbori prin **bootstrap aggregating** (bagging).


<a name='4-1'></a>
### 4.1 - Intelegerea bootstrap aggregating

**Algoritmul Random Forest**:

1. **Bootstrap sampling**: Creeaza $B$ esantioane bootstrap (esantionare aleatoare cu inlocuire)
2. **Selectie aleatoare de trasaturi**: La fiecare split, ia in considerare doar un subset aleator de trasaturi ($\sqrt{n}$ pentru clasificare)
3. **Construieste arborii**: Antreneaza un arbore de decizie pe fiecare esantion bootstrap
4. **Agrega predictiile**: Face media probabilitatilor (clasificare) sau a valorilor (regresie)

**Avantaje cheie**:
- **Reducerea variantei**: Medierea reduce overfitting-ul
- **Eroarea out-of-bag (OOB)**: Validare incorporata folosind exemplele care nu intra in bootstrap
- **Importanta trasaturilor**: Calculata prin mediere peste toti arborii
- **Antrenare paralela**: Arborii sunt independenti

**Hiperparametri**:
- `n_estimators`: Numarul de arbori (mai multi este, de regula, mai bine, dar mai lent)
- `max_features`: Trasaturile luate in considerare la fiecare split
- `max_depth`, `min_samples_split`: Regularizare la nivel de arbore


In [ ]:
# Create a more challenging dataset
X_complex, y_complex = make_classification(
    n_samples=1000, n_features=20, n_informative=15, n_redundant=5,
    n_classes=3, n_clusters_per_class=2, random_state=42
)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_complex, y_complex, test_size=0.3, random_state=42, stratify=y_complex
)

print(f"Complex dataset shape: {X_complex.shape}")
print(f"Number of classes: {len(np.unique(y_complex))}")

<a name='ex-3'></a>
#### **Exercitiul 3 - `train_random_forest`**

**Sarcina ta:**

Implementeaza functia `train_random_forest` care antreneaza un clasificator Random Forest si extrage metrici-cheie, inclusiv eroarea out-of-bag.

Trebuie sa implementezi:

* **Creeaza `RandomForestClassifier`**:
    * Foloseste `RandomForestClassifier` din scikit-learn.
    * Seteaza `n_estimators` (numarul de arbori).
    * Seteaza `max_features` (trasaturi pe split).
    * Seteaza `max_depth` si `min_samples_split` pentru regularizare.
    * Seteaza `oob_score=True` pentru a activa evaluarea out-of-bag.
    * Seteaza `random_state=42` si `n_jobs=-1` (foloseste toate nucleele CPU).

* **Antreneaza modelul**:
    * Antreneaza pe datele oferite.

* **Extrage metricele**:
    * Acuratetea pe test din predictii.
    * Scorul OOB din `model.oob_score_`.
    * Importanta trasaturilor din `model.feature_importances_`.

* **Returneaza rezultatele**:
    * Returneaza un dictionar cu modelul si toate metricele.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru crearea Random Forest:**
* Creeaza: `model = RandomForestClassifier(n_estimators=n_estimators, max_features=max_features, max_depth=max_depth, min_samples_split=min_samples_split, oob_score=True, random_state=42, n_jobs=-1)`.

**Pentru antrenare:**
* Antreneaza: `model.fit(X_train, y_train)`.

**Pentru metrici:**
* Acuratete test: `test_acc = accuracy_score(y_test, model.predict(X_test))`.
* Scor OOB: `oob_score = model.oob_score_`.
* Importanta: `importances = model.feature_importances_`.

**Pentru dictionarul returnat:**
* `results = {'model': model, 'test_acc': test_acc, 'oob_score': oob_score, 'feature_importances': importances}`.

</details>


In [ ]:
# GRADED FUNCTION: train_random_forest
def train_random_forest(X_train, y_train, X_test, y_test, 
                        n_estimators=100, max_features='sqrt', 
                        max_depth=None, min_samples_split=2):
    """
    Train a Random Forest classifier and extract metrics.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        n_estimators: Number of trees
        max_features: Features to consider at each split
        max_depth: Maximum depth of trees
        min_samples_split: Minimum samples to split a node

    Returns:
        results: dict with keys:
            'model': trained RandomForestClassifier
            'test_acc': float - test accuracy
            'oob_score': float - out-of-bag score
            'feature_importances': array of feature importances
    """
    results = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Create RandomForestClassifier
    
    # Fit the model
    
    # Calculate test accuracy
    
    # Get OOB score
    
    # Get feature importances
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Test your implementation
rf_results = train_random_forest(X_train_c, y_train_c, X_test_c, y_test_c,
                                 n_estimators=100, max_features='sqrt', max_depth=15)

print("="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"Test Accuracy: {rf_results['test_acc']:.4f}")
print(f"OOB Score: {rf_results['oob_score']:.4f}")
print(f"Number of trees: {rf_results['model'].n_estimators}")
print("="*60)

# Compare single tree vs Random Forest
single_tree = train_decision_tree(X_train_c, y_train_c, max_depth=15)
single_tree_acc = accuracy_score(y_test_c, single_tree.predict(X_test_c))

print(f"\nComparison:")
print(f"Single Decision Tree: {single_tree_acc:.4f}")
print(f"Random Forest (100 trees): {rf_results['test_acc']:.4f}")
print(f"Improvement: {(rf_results['test_acc'] - single_tree_acc)*100:.2f}%")

##### **Rezultatul asteptat**
```
============================================================
RANDOM FOREST RESULTS
============================================================
Test Accuracy: 0.9XXX
OOB Score: 0.9XXX
Number of trees: 100
============================================================

Comparison:
Single Decision Tree: 0.8XXX
Random Forest (100 trees): 0.9XXX
Improvement: X.XX%
```


In [ ]:
unittests.exercise_3(train_random_forest)

<a name='4-2'></a>
### 4.2 - Analiza importantei trasaturilor

Random Forest ofera estimari robuste ale importantei trasaturilor prin mediere peste toti arborii.


In [ ]:
# Visualize feature importance
importances = rf_results['feature_importances']
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 6))
plt.title("Feature Importances - Random Forest")
plt.bar(range(len(importances)), importances[indices])
plt.xlabel("Feature Index")
plt.ylabel("Importance")
plt.xticks(range(len(importances)), indices)
plt.tight_layout()
plt.show()

print("Top 10 Most Important Features:")
for i in range(min(10, len(importances))):
    print(f"{i+1}. Feature {indices[i]}: {importances[indices[i]]:.4f}")

# Study the effect of number of trees
n_trees_range = [1, 5, 10, 25, 50, 100, 200, 500]
oob_scores = []
test_scores = []

for n_trees in n_trees_range:
    rf_temp = train_random_forest(X_train_c, y_train_c, X_test_c, y_test_c, 
                                   n_estimators=n_trees, max_depth=15)
    oob_scores.append(rf_temp['oob_score'])
    test_scores.append(rf_temp['test_acc'])
    print(f"Trees: {n_trees:3d} | OOB: {rf_temp['oob_score']:.4f} | Test: {rf_temp['test_acc']:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(n_trees_range, oob_scores, marker='o', label='OOB Score', linewidth=2)
plt.plot(n_trees_range, test_scores, marker='s', label='Test Accuracy', linewidth=2)
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest: Performance vs Number of Trees')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

print("\n💡 Key Insight: Performance improves and stabilizes with more trees")

<a name='5'></a>
## 5 - Gradient Boosting

In timp ce Random Forest reduce varianta prin mediere, **Gradient Boosting** reduce bias-ul antrenand secvential arbori care corecteaza erorile arborilor anteriori.


<a name='5-1'></a>
### 5.1 - Cum difera boosting-ul de bagging

**Algoritmul Gradient Boosting**:

1. **Initializeaza**: Porneste cu o predictie simpla (de exemplu, media)
2. **Antrenare secventiala**: Pentru fiecare iteratie $m = 1, ..., M$:
   - Calculeaza reziduurile (erorile) modelului curent
   - Antreneaza un arbore nou pentru a prezice aceste reziduuri
   - Adauga arborele la ensemble cu o rata de invatare: $F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$
3. **Predictia finala**: Suma predictiilor tuturor arborilor

**Diferente-cheie fata de Random Forest**:

| Aspect | Random Forest | Gradient Boosting |
|--------|--------------|-------------------|
| Antrenare | Paralela (arbori independenti) | Secventiala (fiecare arbore depinde de cei anteriori) |
| Obiectiv | Reduce varianta | Reduce bias-ul |
| Arbori | Arbori adanci (bias mic) | Arbori superficiali (bias mare) |
| Viteza | Rapid (paralel) | Mai lent (secvential) |
| Overfitting | Rar | Mai predispus (necesita early stopping) |

**Hiperparametri-cheie**:
- `n_estimators`: Numarul de etape de boosting
- `learning_rate`: Parametrul de shrinkage (mai mic = mai robust, dar necesita mai multi arbori)
- `max_depth`: Adancimea arborelui (de obicei 3-8)
- `subsample`: Fractiunea de exemple pe arbore (stochastic gradient boosting)

**Implementari avansate**:
- **XGBoost**: Boosting regularizat cu performanta mai buna
- **LightGBM**: Gradient-based one-side sampling (GOSS) si exclusive feature bundling (EFB)
- **CatBoost**: Gestioneaza nativ trasaturi categorice


<a name='ex-4'></a>
#### **Exercitiul 4 - `train_gradient_boosting`**

**Sarcina ta:**

Implementeaza functia `train_gradient_boosting` care antreneaza modele gradient boosting cu early stopping.

Trebuie sa implementezi:

* **Creeaza `GradientBoostingClassifier`**:
    * Foloseste `GradientBoostingClassifier` din scikit-learn.
    * Seteaza `n_estimators`, `learning_rate`, `max_depth`, `subsample`.
    * Seteaza `validation_fraction` pentru early stopping.
    * Seteaza `n_iter_no_change` pentru a opri daca nu exista imbunatatire.
    * Seteaza `random_state=42`.

* **Antreneaza cu early stopping**:
    * Antreneaza pe date.
    * Modelul se va opri automat daca scorul pe validare nu se imbunatateste.

* **Extrage metricele**:
    * Acuratetea pe test.
    * Numarul de estimatori folositi efectiv (dupa early stopping).
    * Istoricul de antrenare, daca este disponibil.

* **Optional: antreneaza XGBoost daca este disponibil**:
    * Foloseste `XGBClassifier` cu parametri similari.
    * Foloseste `early_stopping_rounds` pentru early stopping.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru `GradientBoostingClassifier`:**
* Creeaza: `gb_model = GradientBoostingClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, subsample=subsample, validation_fraction=0.1, n_iter_no_change=10, random_state=42)`.
* Antreneaza: `gb_model.fit(X_train, y_train)`.
* Acuratete test: `gb_acc = accuracy_score(y_test, gb_model.predict(X_test))`.
* Arbori folositi: `n_estimators_used = gb_model.n_estimators_`.

**Pentru XGBoost (daca este disponibil):**
* Creeaza: `xgb_model = XGBClassifier(n_estimators=n_estimators, learning_rate=learning_rate, max_depth=max_depth, subsample=subsample, random_state=42, early_stopping_rounds=10)`.
* Antreneaza cu validare: `xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)`.

**Pentru dictionarul returnat:**
* Include toate modelele si metricele in dictionarul de rezultate.

</details>


In [ ]:
# GRADED FUNCTION: train_gradient_boosting
def train_gradient_boosting(X_train, y_train, X_test, y_test,
                           n_estimators=100, learning_rate=0.1, 
                           max_depth=3, subsample=1.0):
    """
    Train gradient boosting models (sklearn and optionally XGBoost).

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        n_estimators: Number of boosting stages
        learning_rate: Learning rate (shrinkage)
        max_depth: Maximum depth of trees
        subsample: Fraction of samples for each tree

    Returns:
        results: dict with keys:
            'gb_model': trained GradientBoostingClassifier
            'gb_acc': test accuracy
            'gb_n_estimators': actual number of estimators used
            'xgb_model': trained XGBClassifier (if available)
            'xgb_acc': test accuracy (if available)
    """
    results = {}
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Train sklearn GradientBoostingClassifier
    
    # Train XGBoost if available
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results

In [ ]:
# Test your implementation
gb_results = train_gradient_boosting(X_train_c, y_train_c, X_test_c, y_test_c,
                                     n_estimators=200, learning_rate=0.1, 
                                     max_depth=3, subsample=0.8)

print("="*60)
print("GRADIENT BOOSTING RESULTS")
print("="*60)
print(f"Gradient Boosting (sklearn) Accuracy: {gb_results['gb_acc']:.4f}")
print(f"Number of estimators used: {gb_results['gb_n_estimators']}")

if 'xgb_acc' in gb_results:
    print(f"XGBoost Accuracy: {gb_results['xgb_acc']:.4f}")
print("="*60)

# Compare with Random Forest
print(f"\nComparison:")
print(f"Random Forest: {rf_results['test_acc']:.4f}")
print(f"Gradient Boosting: {gb_results['gb_acc']:.4f}")

##### **Rezultatul asteptat**
```
============================================================
GRADIENT BOOSTING RESULTS
============================================================
Gradient Boosting (sklearn) Accuracy: 0.9XXX
Number of estimators used: XXX
XGBoost Accuracy: 0.9XXX
============================================================

Comparison:
Random Forest: 0.9XXX
Gradient Boosting: 0.9XXX
```


In [ ]:
unittests.exercise_4(train_gradient_boosting)

In [ ]:
# Study learning rate effect
learning_rates = [0.01, 0.05, 0.1, 0.2, 0.5]
lr_results = []

for lr in learning_rates:
    gb_temp = train_gradient_boosting(X_train_c, y_train_c, X_test_c, y_test_c,
                                      n_estimators=200, learning_rate=lr, max_depth=3)
    lr_results.append({
        'learning_rate': lr,
        'accuracy': gb_temp['gb_acc'],
        'n_estimators': gb_temp['gb_n_estimators']
    })
    print(f"LR: {lr:.2f} | Acc: {gb_temp['gb_acc']:.4f} | Trees: {gb_temp['gb_n_estimators']}")

lr_df = pd.DataFrame(lr_results)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(lr_df['learning_rate'], lr_df['accuracy'], marker='o', linewidth=2)
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Accuracy vs Learning Rate')
axes[0].grid(True, alpha=0.3)

axes[1].plot(lr_df['learning_rate'], lr_df['n_estimators'], marker='s', linewidth=2, color='orange')
axes[1].set_xlabel('Learning Rate')
axes[1].set_ylabel('Number of Trees Used')
axes[1].set_title('Trees Used vs Learning Rate')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight: Lower learning rates need more trees but can generalize better")

<a name='6'></a>
## 6 - Comparatia metodelor ensemble

Hai sa comparam in mod cuprinzator diferite metode ensemble pentru a le intelege compromisurile.


<a name='ex-5'></a>
#### **Exercitiul 5 - `compare_ensemble_methods`**

**Sarcina ta:**

Implementeaza functia `compare_ensemble_methods` care antreneaza si compara mai multe metode ensemble pe acelasi set de date.

Trebuie sa implementezi:

* **Antreneaza mai multe modele**:
    * Un singur arbore de decizie (baseline).
    * Random Forest (bagging).
    * Gradient Boosting (boosting).
    * XGBoost (daca este disponibil).

* **Evalueaza fiecare model**:
    * Acuratete pe test.
    * Timp de antrenare.
    * Timp de predictie.
    * Complexitatea modelului (numar de estimatori/frunze).

* **Returneaza un DataFrame de comparatie**:
    * Cate un rand pentru fiecare model, cu toate metricele.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru antrenare si cronometrare:**
* Foloseste modulul `time`: `import time`.
* Porneste cronometrul: `start = time.time()`.
* Antreneaza modelul.
* Durata: `train_time = time.time() - start`.

**Pentru cronometrarea predictiei:**
* Porneste cronometrul, prezice, masoara durata.

**Pentru complexitate:**
* Arbore de decizie: `model.get_n_leaves()`.
* Random Forest: `model.n_estimators`.
* Gradient Boosting: `model.n_estimators_`.

**Pentru stocarea rezultatelor:**
* Creeaza o lista de dictionare, apoi converteste-o in DataFrame.

</details>


In [ ]:
# GRADED FUNCTION: compare_ensemble_methods
def compare_ensemble_methods(X_train, y_train, X_test, y_test):
    """
    Train and compare multiple ensemble methods.

    Args:
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels

    Returns:
        results_df: DataFrame comparing different methods with columns:
            'method', 'accuracy', 'train_time', 'predict_time', 'complexity'
    """
    import time
    results = []
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Train and evaluate Decision Tree
    
    # Train and evaluate Random Forest
    
    # Train and evaluate Gradient Boosting
    
    # Train and evaluate XGBoost (if available)
    
    # Convert to DataFrame
    
    ### SFÂRȘIT DE COD AICI ###
    
    return results_df

In [ ]:
# Test your implementation
comparison_df = compare_ensemble_methods(X_train_c, y_train_c, X_test_c, y_test_c)

print("="*80)
print("ENSEMBLE METHODS COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Accuracy
axes[0].bar(comparison_df['method'], comparison_df['accuracy'])
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Test Accuracy Comparison')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Training Time
axes[1].bar(comparison_df['method'], comparison_df['train_time'], color='orange')
axes[1].set_ylabel('Time (seconds)')
axes[1].set_title('Training Time Comparison')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3, axis='y')

# Plot 3: Complexity
axes[2].bar(comparison_df['method'], comparison_df['complexity'], color='green')
axes[2].set_ylabel('Complexity (trees/leaves)')
axes[2].set_title('Model Complexity Comparison')
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print("• Random Forest: Best for parallel training, good accuracy")
print("• Gradient Boosting: Often highest accuracy, sequential training")
print("• XGBoost: Optimized boosting, faster than sklearn GradientBoosting")
print("• Single Tree: Fastest but lowest accuracy (high bias)")

##### **Rezultatul asteptat**
```
================================================================================
ENSEMBLE METHODS COMPARISON
================================================================================
           method  accuracy  train_time  predict_time  complexity
 Decision Tree      0.8XXX      0.0XXX        0.00XX          XX
 Random Forest      0.9XXX      0.XXX         0.0XXX         100
Gradient Boosting   0.9XXX      X.XXX         0.0XXX         XXX
          XGBoost   0.9XXX      0.XXX         0.00XX         100
================================================================================
```


In [ ]:
unittests.exercise_5(compare_ensemble_methods)

<a name='7'></a>
## 7 - Arbore de decizie de la zero

Acum hai sa implementam un arbore de decizie de la zero pentru a intelege cu adevarat cum functioneaza algoritmul.


<a name='7-1'></a>
### 7.1 - Implementarea criteriilor de split

Vom implementa atat impuritatea Gini, cat si entropia, pentru a masura calitatea split-urilor.


<a name='ex-6a'></a>
#### **Exercitiul 6a - `calculate_gini`**

**Sarcina ta:**

Implementeaza functia `calculate_gini` care calculeaza impuritatea Gini pentru un set de etichete.

Trebuie sa implementezi:

* **Calculeaza probabilitatile claselor**:
    * Numara exemplele din fiecare clasa.
    * Imparte la numarul total de exemple pentru a obtine probabilitatile.

* **Calculeaza impuritatea Gini**:
    * Foloseste formula: $\text{Gini} = 1 - \sum_{i=1}^{C} p_i^2$
    * Returneaza 0 daca toate exemplele sunt intr-o singura clasa (nod pur).

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru probabilitatile claselor:**
* Obtine clasele unice si frecventele: `classes, counts = np.unique(y, return_counts=True)`.
* Calculeaza probabilitatile: `probabilities = counts / len(y)`.

**Pentru Gini:**
* Suma patratelor: `gini = 1 - np.sum(probabilities ** 2)`.
* Sau cu bucla: `gini = 1 - sum([p**2 for p in probabilities])`.

</details>


In [ ]:
# GRADED FUNCTION: calculate_gini
def calculate_gini(y):
    """
    Calculate Gini impurity for a set of labels.

    Args:
        y: numpy array of labels

    Returns:
        gini: float - Gini impurity (0 = pure, higher = more impure)
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Handle empty array
    if len(y) == 0:
        return 0
    
    # Calculate class probabilities
    
    # Calculate Gini impurity
    
    ### SFÂRȘIT DE COD AICI ###
    
    return gini

In [ ]:
# Test calculate_gini
test_cases = [
    np.array([0, 0, 0, 0]),           # Pure node
    np.array([0, 1, 0, 1]),           # Maximum impurity (binary)
    np.array([0, 0, 1, 1, 2, 2]),     # Equal distribution (3 classes)
    np.array([0, 0, 0, 1, 1, 2]),     # Unequal distribution
]

print("Gini Impurity Test Cases:")
for i, y_case in enumerate(test_cases):
    gini = calculate_gini(y_case)
    print(f"Case {i+1}: {y_case } -> Gini = {gini:.4f}")

##### **Rezultatul asteptat**
```
Gini Impurity Test Cases:
Case 1: [0 0 0 0] -> Gini = 0.0000
Case 2: [0 1 0 1] -> Gini = 0.5000
Case 3: [0 0 1 1 2 2] -> Gini = 0.6667
Case 4: [0 0 0 1 1 2] -> Gini = 0.6111
```


In [ ]:
unittests.exercise_6a(calculate_gini)

<a name='ex-6b'></a>
#### **Exercitiul 6b - `calculate_entropy`**

**Sarcina ta:**

Implementeaza functia `calculate_entropy` care calculeaza entropia pentru un set de etichete.

Trebuie sa implementezi:

* **Calculeaza probabilitatile claselor**:
    * La fel ca la Gini: numara exemplele din fiecare clasa si normalizeaza.

* **Calculeaza entropia**:
    * Foloseste formula: $\text{Entropy} = -\sum_{i=1}^{C} p_i \log_2(p_i)$
    * Gestioneaza cazul $p_i = 0$ (foloseste conventia $0 \log 0 = 0$).

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru entropie:**
* Filtreaza probabilitatile zero: `probs = probabilities[probabilities > 0]`.
* Calculeaza entropia: `entropy = -np.sum(probs * np.log2(probs))`.
* Sau cu bucla: `entropy = -sum([p * np.log2(p) for p in probs if p > 0])`.

</details>


In [ ]:
# GRADED FUNCTION: calculate_entropy
def calculate_entropy(y):
    """
    Calculate entropy for a set of labels.

    Args:
        y: numpy array of labels

    Returns:
        entropy: float - entropy (0 = pure, higher = more impure)
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Handle empty array
    if len(y) == 0:
        return 0
    
    # Calculate class probabilities
    
    # Calculate entropy (handle log(0) = 0)
    
    ### SFÂRȘIT DE COD AICI ###
    
    return entropy

In [ ]:
# Test calculate_entropy
print("Entropy Test Cases:")
for i, y_case in enumerate(test_cases):
    entropy = calculate_entropy(y_case)
    gini = calculate_gini(y_case)
    print(f"Case {i+1}: {y_case }")
    print(f"  Entropy = {entropy:.4f}, Gini = {gini:.4f}")

##### **Rezultatul asteptat**
```
Entropy Test Cases:
Case 1: [0 0 0 0]
  Entropy = 0.0000, Gini = 0.0000
Case 2: [0 1 0 1]
  Entropy = 1.0000, Gini = 0.5000
Case 3: [0 0 1 1 2 2]
  Entropy = 1.5850, Gini = 0.6667
Case 4: [0 0 0 1 1 2]
  Entropy = 1.4591, Gini = 0.6111
```


In [ ]:
unittests.exercise_6b(calculate_entropy)

<a name='ex-6c'></a>
#### **Exercitiul 6c - `find_best_split`**

**Sarcina ta:**

Implementeaza functia `find_best_split` care gaseste cea mai buna trasatura si cel mai bun prag pentru a split-ui un nod.

Trebuie sa implementezi:

* **Parcurge toate trasaturile**:
    * Pentru fiecare trasatura, incearca valori diferite de prag.

* **Pentru fiecare split potential**:
    * Imparte datele in subseturi stanga si dreapta.
    * Calculeaza impuritatea ponderata a split-ului.
    * Urmareste split-ul cu impuritatea minima.

* **Calculeaza information gain**:
    * Information gain = impuritatea parintelui - impuritatea ponderata a copiilor.

* **Returneaza cel mai bun split**:
    * Returneaza indicele trasaturii, pragul si information gain-ul.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru praguri:**
* Foloseste valorile unice: `thresholds = np.unique(X[:, feature_idx])`.
* Sau punctele de mijloc dintre valorile sortate.

**Pentru split:**
* Masca stanga: `left_mask = X[:, feature_idx] <= threshold`.
* Masca dreapta: `right_mask = ~left_mask`.
* Etichete split-uite: `y_left = y[left_mask]`, `y_right = y[right_mask]`.

**Pentru impuritatea ponderata:**
* Ponderi: `w_left = len(y_left) / len(y)`, `w_right = len(y_right) / len(y)`.
* Impuritate ponderata: `weighted_impurity = w_left * calculate_gini(y_left) + w_right * calculate_gini(y_right)`.

**Pentru information gain:**
* Impuritatea parintelui: `parent_impurity = calculate_gini(y)`.
* Gain: `gain = parent_impurity - weighted_impurity`.

</details>


In [ ]:
# GRADED FUNCTION: find_best_split
def find_best_split(X, y, criterion='gini'):
    """
    Find the best feature and threshold to split the data.

    Args:
        X: numpy array of shape (n_samples, n_features)
        y: numpy array of labels
        criterion: 'gini' or 'entropy'

    Returns:
        best_feature: int - index of best feature to split on
        best_threshold: float - best threshold value
        best_gain: float - information gain from this split
    """
    best_gain = 0
    best_feature = None
    best_threshold = None
    
    # Choose impurity function
    impurity_func = calculate_gini if criterion == 'gini' else calculate_entropy
    parent_impurity = impurity_func(y)
    
    n_samples, n_features = X.shape
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Loop through all features
    
        # Get unique values as potential thresholds
        
        # Try each threshold
        
            # Split data
            
            # Skip if split doesn't separate data
            
            # Calculate weighted impurity
            
            # Calculate information gain
            
            # Update best split if this is better
            
    ### SFÂRȘIT DE COD AICI ###
    
    return best_feature, best_threshold, best_gain

In [ ]:
# Test find_best_split
X_test_split = np.array([
    [1, 2],
    [2, 3],
    [3, 4],
    [6, 7],
    [7, 8],
    [8, 9]
])
y_test_split = np.array([0, 0, 0, 1, 1, 1])

feature, threshold, gain = find_best_split(X_test_split, y_test_split, criterion='gini')
print(f"Best split: Feature {feature}, Threshold {threshold:.2f}, Gain {gain:.4f}")

# Verify the split
left_mask = X_test_split[:, feature] <= threshold
print(f"\nLeft subset labels: {y_test_split[left_mask]}")
print(f"Right subset labels: {y_test_split[~left_mask]}")
print("(Should be perfectly separated)")

##### **Rezultatul asteptat**
```
Best split: Feature 0, Threshold X.XX, Gain X.XXXX

Left subset labels: [0 0 0]
Right subset labels: [1 1 1]
(Should be perfectly separated)
```


In [ ]:
unittests.exercise_6c(find_best_split)

<a name='ex-6d'></a>
#### **Exercitiul 6d - `build_decision_tree_scratch`**

**Sarcina ta:**

Implementeaza functia `build_decision_tree_scratch` care construieste recursiv un arbore de decizie.

Trebuie sa implementezi:

* **Verifica conditiile de oprire**:
    * S-a atins adancimea maxima.
    * Nodul este pur (toate exemplele apartin aceleiasi clase).
    * Sunt prea putine exemple pentru split.
    * Split-ul nu aduce information gain.

* **Daca se opreste, creeaza un nod frunza**:
    * Returneaza un dictionar cu indicatorul `'leaf'` si clasa majoritara.

* **Altfel, gaseste cel mai bun split**:
    * Foloseste `find_best_split` pentru a obtine trasatura si pragul.

* **Construieste recursiv subarborii**:
    * Imparte datele pe baza trasaturii si pragului.
    * Construieste recursiv subarborele stang si drept.

* **Returneaza nodul intern**:
    * Stocheaza trasatura, pragul si copiii stanga/dreapta.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru conditiile de oprire:**
* Verifica adancimea: `if depth >= max_depth or len(y) < min_samples_split`.
* Verifica puritatea: `if len(np.unique(y)) == 1`.
* Gaseste cel mai bun split si verifica gain-ul: `if best_gain == 0`.

**Pentru nodul frunza:**
* Clasa majoritara: `majority = np.bincount(y).argmax()`.
* Returneaza: `return {'leaf': True, 'class': majority}`.

**Pentru nodul intern:**
* Split-uie datele si apeleaza recursiv:
  ```python
  left_mask = X[:, best_feature] <= best_threshold
  left = build_decision_tree_scratch(X[left_mask], y[left_mask], depth+1, ...)
  right = build_decision_tree_scratch(X[~left_mask], y[~left_mask], depth+1, ...)
  return {'leaf': False, 'feature': best_feature, 'threshold': best_threshold,
          'left': left, 'right': right}
  ```

</details>


In [ ]:
# GRADED FUNCTION: build_decision_tree_scratch
def build_decision_tree_scratch(X, y, depth=0, max_depth=5, 
                               min_samples_split=2, criterion='gini'):
    """
    Recursively build a decision tree.

    Args:
        X: numpy array of features
        y: numpy array of labels
        depth: current depth
        max_depth: maximum depth allowed
        min_samples_split: minimum samples to split
        criterion: 'gini' or 'entropy'

    Returns:
        tree: dict representing the tree structure
            Leaf node: {'leaf': True, 'class': int}
            Internal node: {'leaf': False, 'feature': int, 'threshold': float,
                           'left': tree, 'right': tree}
    """
    ### ÎNCEPUT DE COD AICI ###
    
    # Check stopping conditions
    
    # If stopping, return leaf node
    
    # Find best split
    
    # If no good split, return leaf
    
    # Split data
    
    # Recursively build left and right subtrees
    
    # Return internal node
    
    ### SFÂRȘIT DE COD AICI ###
    
    return tree

In [ ]:
def predict_tree_scratch(tree, x):
    """Predict class for a single sample using the tree."""
    if tree['leaf']:
        return tree['class']
    
    if x[tree['feature']] <= tree['threshold']:
        return predict_tree_scratch(tree['left'], x)
    else:
        return predict_tree_scratch(tree['right'], x)

def predict_batch_scratch(tree, X):
    """Predict classes for multiple samples."""
    return np.array([predict_tree_scratch(tree, x) for x in X])

# Test the decision tree from scratch
X_train_simple = X_train[:, [2, 3]]  # Use 2 features for simplicity
tree_scratch = build_decision_tree_scratch(X_train_simple, y_train, 
                                          max_depth=3, criterion='gini')

# Make predictions
y_pred_scratch = predict_batch_scratch(tree_scratch, X_test[:, [2, 3]])
scratch_acc = accuracy_score(y_test, y_pred_scratch)

print(f"Decision Tree from Scratch Accuracy: {scratch_acc:.4f}")

# Compare with sklearn
sklearn_tree = train_decision_tree(X_train_simple, y_train, max_depth=3)
sklearn_acc = accuracy_score(y_test, sklearn_tree.predict(X_test[:, [2, 3]]))
print(f"Sklearn Decision Tree Accuracy: {sklearn_acc:.4f}")

# Visualize decision boundary
class TreeWrapper:
    def __init__(self, tree):
        self.tree = tree
    def predict(self, X):
        return predict_batch_scratch(self.tree, X)

plot_decision_boundary_2d(TreeWrapper(tree_scratch), X_train_simple, y_train,
                         title="From-Scratch Decision Tree Boundary")

##### **Rezultatul asteptat**
```
Decision Tree from Scratch Accuracy: 0.9XXX
Sklearn Decision Tree Accuracy: 0.9XXX
```


In [ ]:
unittests.exercise_6d(build_decision_tree_scratch)

<a name='8'></a>
## 8 - Random Forest de la zero

In final, hai sa implementam Random Forest combinand mai multi arbori de decizie cu bootstrap sampling.


<a name='ex-7'></a>
#### **Exercitiul 7 - `random_forest_from_scratch`**

**Sarcina ta:**

Implementeaza functia `random_forest_from_scratch` care construieste un ensemble de tip random forest.

Trebuie sa implementezi:

* **Bootstrap sampling**:
    * Pentru fiecare arbore, creeaza un esantion bootstrap (esantionare aleatoare cu inlocuire).
    * Esantioneaza `n_samples` linii din date (cu inlocuire).

* **Subesantionarea trasaturilor**:
    * La fiecare split, selecteaza aleator un subset de trasaturi luate in considerare.
    * Tipic: `sqrt(n_features)` pentru clasificare.

* **Construieste mai multi arbori**:
    * Foloseste `build_decision_tree_scratch` pentru fiecare esantion bootstrap.
    * Stocheaza toti arborii intr-o lista.

* **Agrega predictiile**:
    * La predictie, fiecare arbore voteaza.
    * Returneaza clasa majoritara peste toti arborii.

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (apasati pentru a extinde daca te-ai blocat)</font></b></summary>

Daca ai nevoie de putin ajutor:

**Pentru bootstrap sampling:**
* Indici aleatori: `indices = np.random.choice(len(X), size=len(X), replace=True)`.
* Esantion bootstrap: `X_boot = X[indices]`, `y_boot = y[indices]`.

**Pentru construirea padurii:**
* Parcurge: `for i in range(n_estimators):`.
* Creeaza esantionul bootstrap.
* Construieste arborele: `tree = build_decision_tree_scratch(X_boot, y_boot, ...)`.
* Stocheaza-l: `trees.append(tree)`.

**Pentru predictie:**
* Obtine toate predictiile: `all_preds = [predict_batch_scratch(tree, X) for tree in trees]`.
* Stivuieste-le: `all_preds = np.array(all_preds)`.
* Voteaza: `predictions = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=all_preds)`.

**Nota**: Pentru simplitate, aceasta implementare nu include subesantionarea trasaturilor la fiecare split. Acest lucru ar necesita modificarea lui `find_best_split` astfel incat sa accepte un subset de trasaturi.

</details>


In [ ]:
# GRADED FUNCTION: random_forest_from_scratch
def random_forest_from_scratch(X, y, n_estimators=10, max_depth=5, 
                               min_samples_split=2, criterion='gini'):
    """
    Build a random forest by training multiple decision trees on bootstrap samples.

    Args:
        X: Training features
        y: Training labels
        n_estimators: Number of trees in the forest
        max_depth: Maximum depth of each tree
        min_samples_split: Minimum samples to split a node
        criterion: 'gini' or 'entropy'

    Returns:
        forest: list of decision trees
    """
    forest = []
    n_samples = len(X)
    
    ### ÎNCEPUT DE COD AICI ###
    
    # Build each tree
    
        # Create bootstrap sample
        
        # Build decision tree on bootstrap sample
        
        # Add to forest
        
    ### SFÂRȘIT DE COD AICI ###
    
    return forest

def predict_forest_scratch(forest, X):
    """
    Predict classes using a random forest (majority voting).
    
    Args:
        forest: list of decision trees
        X: features to predict on
        
    Returns:
        predictions: numpy array of predicted classes
    """
    # Get predictions from all trees
    all_predictions = np.array([predict_batch_scratch(tree, X) for tree in forest])
    
    # Majority vote
    predictions = np.apply_along_axis(
        lambda x: np.bincount(x.astype(int)).argmax(), 
        axis=0, 
        arr=all_predictions
    )
    
    return predictions

In [ ]:
# Test Random Forest from scratch
print("Training Random Forest from scratch...")
forest_scratch = random_forest_from_scratch(X_train_simple, y_train, 
                                           n_estimators=20, max_depth=5, 
                                           criterion='gini')

# Make predictions
y_pred_forest = predict_forest_scratch(forest_scratch, X_test[:, [2, 3]])
forest_scratch_acc = accuracy_score(y_test, y_pred_forest)

print(f"\nRandom Forest from Scratch Accuracy: {forest_scratch_acc:.4f}")
print(f"Single Tree from Scratch Accuracy: {scratch_acc:.4f}")
print(f"Improvement: {(forest_scratch_acc - scratch_acc)*100:.2f}%")

# Compare with sklearn
sklearn_rf = train_random_forest(X_train_simple, y_train, X_test[:, [2, 3]], y_test,
                                 n_estimators=20, max_depth=5)
print(f"Sklearn Random Forest Accuracy: {sklearn_rf['test_acc']:.4f}")

# Visualize Random Forest decision boundary
class ForestWrapper:
    def __init__(self, forest):
        self.forest = forest
    def predict(self, X):
        return predict_forest_scratch(self.forest, X)

plot_decision_boundary_2d(ForestWrapper(forest_scratch), X_train_simple, y_train,
                         title="From-Scratch Random Forest Boundary (20 trees)")

print(f"\n Random Forest combines {len(forest_scratch)} trees to make robust predictions!")

##### **Rezultatul asteptat**
```
Training Random Forest from scratch...

Random Forest from Scratch Accuracy: 0.9XXX
Single Tree from Scratch Accuracy: 0.9XXX
Improvement: X.XX%
Sklearn Random Forest Accuracy: 0.9XXX

Random Forest combines 20 trees to make robust predictions!
```


In [ ]:
unittests.exercise_7(random_forest_from_scratch)

## Felicitari!

Ai finalizat cu succes tema despre Modele Bazate pe Arbori! Ai construit o intelegere cuprinzatoare a uneia dintre cele mai puternice si mai utilizate familii de algoritmi de machine learning.

### Ce ai realizat:

1. **Ai inteles arborii de decizie**: Ai invatat despre criteriile de split, information gain si procesul de construire a arborelui
2. **Ai antrenat si vizualizat arbori**: Ai construit arbori de decizie interpretabili si ai extras reguli relevante
3. **Ai controlat overfitting-ul**: Ai aplicat tehnici de pruning si ai comparat efectele lor
4. **Ai implementat Random Forest**: Ai folosit bootstrap aggregating pentru a reduce varianta si a imbunatati predictiile
5. **Ai analizat importanta trasaturilor**: Ai extras si interpretat importanta trasaturilor din modele ensemble
6. **Ai aplicat Gradient Boosting**: Ai antrenat XGBoost si LightGBM cu early stopping
7. **Ai comparat metode ensemble**: Ai inteles compromisurile dintre bagging si boosting
8. **Ai construit arbori de la zero**: Ai implementat un arbore de decizie complet cu split-uri pe baza de Gini/entropie
9. **Ai construit Random Forest de la zero**: Ai implementat mecanismele de bootstrap sampling si votare

### Idei-cheie:

* **Arborii de decizie sunt intuitivi**: Usor de vizualizat, interpretat si explicat stakeholderilor non-tehnici
* **Arborii individuali fac usor overfitting**: Fara regularizare, memoreaza datele de train
* **Ensemble-urile sunt puternice**: Combinarea arborilor imbunatateste dramatic performanta
* **Random Forest (Bagging)**: Reduce varianta prin mediere, se antreneaza in paralel
* **Gradient Boosting**: Reduce bias-ul prin corectie secventiala, obtinand adesea cea mai buna acuratete
* **Importanta trasaturilor**: Modelele bazate pe arbori ofera in mod natural ranking-uri de trasaturi
* **Nu este nevoie de scalarea trasaturilor**: Spre deosebire de multi algoritmi, arborii nu necesita normalizare

### Cand sa folosesti modele bazate pe arbori:

**Excelente pentru:**
- Date tabelare/structurate (CSV, baze de date)
- Tipuri mixte de trasaturi (numerice + categorice)
- Relatii neliniare
- Interactiuni intre trasaturi
- Modele interpretabile (arbori individuali)
- Competitii (XGBoost/LightGBM)
- Gestionarea valorilor lipsa

**Ia in considerare alternative pentru:**
- Date nestructurate (imagini, text, audio) - foloseste deep learning
- Date rare cu dimensionalitate foarte mare - ia in calcul modele liniare
- Situatii in care ai nevoie de calibrarea probabilitatilor - necesita pasi suplimentari
- Predictie in timp real cu constrangeri stricte de latenta - arborii individuali sunt mai potriviti decat ensemble-urile mari

### Ghid de selectie a modelului:

**Decision Tree:**
- Foloseste-l cand: interpretabilitatea este critica, ai nevoie de un baseline simplu
- Avantaje: rapid, interpretabil, fara preprocesare
- Dezavantaje: varianta mare, face usor overfitting

**Random Forest:**
- Foloseste-l cand: vrei robustete, antrenare paralela, o alegere buna by default
- Avantaje: varianta mica, gestioneaza outlieri, validare OOB
- Dezavantaje: mai putin interpretabil, consum mai mare de memorie

**Gradient Boosting (XGBoost/LightGBM):**
- Foloseste-l cand: acuratetea maxima este prioritatea, inclusiv in competitii
- Avantaje: adesea cea mai buna performanta, gestioneaza valorile lipsa
- Dezavantaje: antrenare mai lenta, mai multi hiperparametri, predispus la overfitting

### Sfaturi pentru ajustarea hiperparametrilor:

**Pentru arborii de decizie:**
- `max_depth`: Incepe cu 3-5, creste daca vezi underfitting
- `min_samples_split`: 2-20, in functie de dimensiunea setului de date
- `criterion`: Incearca atat `'gini'`, cat si `'entropy'`

**Pentru Random Forest:**
- `n_estimators`: 100-500 (mai multi este, de regula, mai bine)
- `max_features`: `'sqrt'` pentru clasificare, `'0.3'` pentru regresie
- `max_depth`: 10-30 sau `None`

**Pentru Gradient Boosting:**
- `n_estimators`: 100-1000 cu early stopping
- `learning_rate`: 0.01-0.1 (mai mic = mai multi arbori necesari)
- `max_depth`: 3-8 (arborii superficiali functioneaza cel mai bine)
- `subsample`: 0.7-0.9 pentru regularizare

**Excelent! Acum ai o intelegere de nivel avansat a modelelor bazate pe arbori!**
